# Bureau.csv Analysis
- all credits provided to our clinets that were reported to Credit Bureau
- 1 row per client
---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
bureau_df = pd.read_csv("../data/raw/bureau.csv")
print(f"Shape of bureau_df: {bureau_df.shape}")
bureau_df.head()

Shape of bureau_df: (1716428, 17)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


## 2. Review the columns

General credit info data
- `SK_ID_CURR` - ID of loan in our sample - one loan in our sample can have 0,1,2 or more related previous credits in credit bureau
- `SK_ID_BUREAU` - Recoded ID of previous Credit Bureau credit related to our loan (unique coding for each loan application)
- `CREDIT_ACTIVE` - Status of the Credit Bureau (CB) reported credits
- `CREDIT_CURRENCY` - Recoded currency of the Credit Bureau credit, recoded
- `CNT_CREDIT_PROLONG` - How many times was the Credit Bureau credit prolonged
- `CREDIT_TYPE` - Type of Credit Bureau credit (Car, cash,...)

In [14]:
type_of_cols_in_bureau = []
general_cols = [
    'SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE',
    'CREDIT_CURRENCY', 'CNT_CREDIT_PROLONG', 'CREDIT_TYPE'
]
type_of_cols_in_bureau.append(general_cols)
print(f"Number of general columns: {len(general_cols)}")

Number of general columns: 6


Overdues and amounts data
- `AMT_CREDIT_MAX_OVERDUE` - Maximal amount overdue on the Credit Bureau credit so far (at application date of loan in our sample)
- `AMT_CREDIT_SUM` - Current credit amount for the Credit Bureau credit
- `AMT_CREDIT_SUM_DEBT` - Current debt on Credit Bureau credit
- `AMT_CREDIT_SUM_LIMIT` - Current credit limit of credit card reported in Credit Bureau
- `AMT_CREDIT_SUM_OVERDUE` - Current amount overdue on Credit Bureau credit

In [17]:
overdues_amounts_cols = [
    'AMT_CREDIT_MAX_OVERDUE' , 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT',
    'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE'  
]
type_of_cols_in_bureau.append(overdues_amounts_cols)
print(f"Number of overdues_amounts columns: {len(overdues_amounts_cols)}")

Number of overdues_amounts columns: 5


Time data
- `DAYS_CREDIT` - How many days before current application did client apply for Credit Bureau credit,time only relative to the application
- `CREDIT_DAY_OVERDUE` - Number of days past due on CB credit at the time of application for related loan in our sample
- `DAYS_CREDIT_ENDDATE` - Remaining duration of CB credit (in days) at the time of application in Home Credit,time only relative to the application
- `DAYS_ENDDATE_FACT` - Days since CB credit ended at the time of application in Home Credit (only for closed credit),time only relative to the application
- `DAYS_CREDIT_UPDATE` - How many days before loan application did last information about the Credit Bureau credit come,time only relative to the application

In [18]:
time_cols = [
    'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE',
    'DAYS_ENDDATE_FACT', 'DAYS_CREDIT_UPDATE'
]
type_of_cols_in_bureau.append(time_cols)
print(f"Number of time columns: {len(time_cols)}")

Number of time columns: 5


## 3. Sum of missing values in columns

In [19]:
bureau_n = bureau_df.shape[0]
print('------------------------')
for cols in type_of_cols_in_bureau:
    print(bureau_df[cols].isna().sum() / bureau_n)
    print('------------------------')

------------------------
SK_ID_CURR            0.0
SK_ID_BUREAU          0.0
CREDIT_ACTIVE         0.0
CREDIT_CURRENCY       0.0
CNT_CREDIT_PROLONG    0.0
CREDIT_TYPE           0.0
dtype: float64
------------------------
AMT_CREDIT_MAX_OVERDUE    0.655133
AMT_CREDIT_SUM            0.000008
AMT_CREDIT_SUM_DEBT       0.150119
AMT_CREDIT_SUM_LIMIT      0.344774
AMT_CREDIT_SUM_OVERDUE    0.000000
dtype: float64
------------------------
DAYS_CREDIT            0.000000
CREDIT_DAY_OVERDUE     0.000000
DAYS_CREDIT_ENDDATE    0.061496
DAYS_ENDDATE_FACT      0.369170
DAYS_CREDIT_UPDATE     0.000000
dtype: float64
------------------------


- `AMT_CREDIT_MAX_OVERDUE` - 65.5% - most of the values are missing, this column can be important so we can consider create binary variable `AMT_CREDIT_MAX_OVERDUE_MISSING`
- `AMT_CREDIT_SUM_DEBT` - 15% - we can try to impute this column
- `AMT_CREDIT_SUM_LIMIT` - 34.5% -
- `DAYS_CREDIT_ENDDATE` - 6.1% - 
- `DAYS_ENDDATE_FACT` - 37% -

In [20]:
bureau_df['AMT_CREDIT_MAX_OVERDUE'].describe()

count    5.919400e+05
mean     3.825418e+03
std      2.060316e+05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.159872e+08
Name: AMT_CREDIT_MAX_OVERDUE, dtype: float64